# Net Benefit (Decision Curve Analysis)

## Set Up

libraries/packages

In [1]:
import sys
import os

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
import matplotlib.pyplot as plt
from statkit.decision import NetBenefitDisplay
from src.data_utils import get_data, get_models
import pandas as pd

Data/Models

In [ ]:
# Data
DATA_DICT = {
    "base": get_data(is_nomo=False),
    "nomo": get_data(is_nomo=True),
}


# Models
model_dir = BASE_PATH / "artifacts/models/trained"
model_prefix_list = ["lgbm", "xgb", "knn", "svc", "nn", "stack"]
## Base models
model_dict = get_models(model_prefix_list, model_dir)
## Nomogram
model_dict.update(get_models(["lr"], model_dir))

Construct plot

In [ ]:
color_list = [
    "tab:blue",
    "tab:orange",
    "tab:green",
    "tab:purple",
    "tab:red",
    "tab:olive",
    "tab:pink",
    "tab:gray",
    "tab:olive",
    "tab:cyan",
]
plt.figure(figsize=(10, 8))
ax = plt.gca()
for (model_name, model), color in zip(model_dict.items(), color_list):
    keyword = "nomo" if model_name == "lr" else "base"
    X_test = DATA_DICT[keyword]["X"]
    y_test = DATA_DICT[keyword]["y"].values.ravel()
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    NetBenefitDisplay.from_predictions(
        y_test,
        y_pred_proba,
        name=model_name,
        ax=ax,
    )
model_names = list(model_dict.keys())
lines = [line for line in ax.get_lines() if line.get_label() in model_names]
for line, color in zip(lines, color_list):
    line.set_color(color)


# Get all handles and labels
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))  # Remove duplicates

# # Separate baselines from models
baseline_labels = ["Always act", "Never act", "Oracle"]
model_labels = [name for name in model_names if name in by_label]

# # Models first, then baselines
ordered_labels = model_labels + baseline_labels

# # Create ordered handles and labels
ordered_handles = [by_label[label] for label in ordered_labels if label in by_label]
final_labels = [label for label in ordered_labels if label in by_label]

ax.legend(ordered_handles, final_labels, loc="upper right")

plt.title("Decision Curve Analysis: Model Comparison")
plt.ylim(-0.2, 0.6)


dca_path = BASE_PATH / "results/figures/DCA.pdf"
if dca_path.exists():
    dca_path.unlink()
dca_path.parent.mkdir(exist_ok=True, parents=True)
plt.savefig(dca_path, bbox_inches="tight")